In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("Bengaluru_House_Data.csv")

In [ ]:
pd.set_option("display.max_columns",None)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape

Data Filtering

In [ ]:
df = df.drop(columns=["area_type","society"])

In [ ]:
df['bhk']=df["size"].str.split().str[0].astype(float)

In [ ]:
df.dropna(subset=['total_sqft'], inplace=True)

In [ ]:
def convert_sqft_to_num(x):
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
df = df.dropna()

In [ ]:
df['per_sqft']=df['price']*100000/df['total_sqft']

In [ ]:
df.drop(columns=['size','availability'],inplace=True)

In [ ]:
def removing_outliers(df):
    err_idx= np.array([])
    for loc,loc_df in df.groupby('location'):
        bhk_stats = {}
        for bhk,bhk_df in loc_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean' : bhk_df['per_sqft'].mean(),
                'std' : bhk_df['per_sqft'].std(),
                'count' : bhk_df['per_sqft'].count()
            }
        for bhk,bhk_df in loc_df.groupby('bhk'):
            stats = bhk_stats.get(bhk-1)
            if stats and stats['count']>5:
                outliers = bhk_df[bhk_df['per_sqft'] < stats['mean']-stats['std']].index.values
                err_idx = np.append(err_idx, outliers)
    return df.drop(err_idx,axis='index')
    
df = removing_outliers(df)

In [ ]:
df = df[df.groupby('location')['location'].transform('count') > 10]

In [ ]:
df.drop(columns=['per_sqft'],inplace=True)

Making Model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [ ]:
x = df.drop(columns=['price'])
y = df['price']
x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=400
)

In [ ]:
col_encoded = ['location']
numerical_cols = ['total_sqft','bhk','bath','balcony']
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat', OneHotEncoder(sparse_output=False,handle_unknown='ignore'), col_encoded
        )
    ],
    remainder='passthrough'
)

In [ ]:
pipe_lr = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model', LinearRegression())
])
pipe_rid = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model', Ridge())
])
pipe_rnd = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model', RandomForestRegressor())
])

In [ ]:
pipe_lr.fit(x_train,y_train)
pipe_rid.fit(x_train,y_train)
pipe_rnd.fit(x_train,y_train)

Checking scores

In [ ]:
from sklearn.model_selection import cross_val_score

lr_cv_scores = cross_val_score(pipe_lr, x, y, cv=5, scoring='r2')
print("Linear Regression CV R2 Scores:", lr_cv_scores)
print(f"Mean LR R2 Score: {lr_cv_scores.mean():.4f}")

rnd_cv_scores = cross_val_score(pipe_rnd, x, y, cv=5, scoring='r2')
print("\nRandom Forest CV R2 Scores:", rnd_cv_scores)
print(f"Mean Random Forest R2 Score: {rnd_cv_scores.mean():.4f}")

rid_cv_scores = cross_val_score(pipe_rid, x, y, cv=5, scoring='r2')
print("\nRidge CV R2 Scores:", rid_cv_scores)
print(f"Mean Ridge R2 Score: {rid_cv_scores.mean():.4f}")

Final Model and exporting it

In [ ]:
model = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model', RandomForestRegressor())
])

In [ ]:
model.fit(x,y)

In [ ]:
import pickle

In [ ]:
with open('House_Predicting_Model.pickle','wb') as f:
    pickle.dump(model,f)

print('Model exported...')